YouTube-RAG: Conversational Video Intelligence

Developed by: Nahid Parveen Khanam

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Install all requirements at once
!pip install -U langchain-huggingface langchain-community langchain-experimental langchain-core
!pip install -U sentence-transformers faiss-gpu-cu12 yt-dlp accelerate
!pip install git+https://github.com/openai/whisper.git

import os
import torch
import whisper
from transformers import pipeline
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata

# Set HF Token
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
print("Environment Ready!")

In [ ]:
import os
video_url = "https://www.youtube.com/watch?v=HgcoFVqG0ms"
save_dir = "/content/drive/MyDrive/github/youtube-rag/rag/"
audio_path = os.path.join(save_dir, "audio_HgcoFVqG0ms.m4a")
transcript_path = os.path.join(save_dir, "transcript_HgcoFVqG0ms.txt")

os.makedirs(save_dir, exist_ok=True)

print("Downloading audio...")
!yt-dlp -x --audio-format m4a -o "{audio_path}" "{video_url}"

print("Transcribing... (Stay on this page)")
model_whisper = whisper.load_model("base")
result = model_whisper.transcribe(audio_path)
transcript_text = result["text"]

with open(transcript_path, "w") as f:
    f.write(transcript_text)

print(f"Transcript Saved! Length: {len(transcript_text)} characters.")

In [ ]:
os.environ["SENTENCE_TRANSFORMERS_HOME"] = "/content/cache"
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={'device': 'cuda'})

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=70)
docs = text_splitter.create_documents([transcript_text])

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"Success! {len(docs)} chunks indexed.")

In [ ]:
model_id = "HuggingFaceH4/zephyr-7b-beta"
pipe = pipeline("text-generation", model=model_id, torch_dtype=torch.float16, device_map="auto", max_new_tokens=512, temperature=0.1)
llm = HuggingFacePipeline(pipeline=pipe)

prompt = ChatPromptTemplate.from_template("<|system|>You are a helpful assistant. Use the following context to answer. Context: {context} </s> <|user|>{question} </s> <|assistant|>")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser())

query = "What is the difference between LangChain and LangGraph?"
print(f"ANSWER: {rag_chain.invoke(query)}")
vectorstore.save_local(os.path.join(save_dir, "faiss_index"))